In [29]:
""" tombo preprocess annotate_raw_with_fastq \
    --fast5-basedir /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy_single \
    --fastq-filenames /root/sunxh/WaveCrossMamba/contrast/IVT_normalA.fastq \
    --overwrite
tombo detect_modifications de_novo     --fast5-basedir /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single  --statistics-file-basename tombo_m6A --rna --processes 16
tombo stats display_stats --per-read-statistics /root/tombo_m6A.tombo.stats.per_read_stats
tombo detect_modifications de_novo \
    --fast5-basedir /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single \
    --statistics-file-basename tombo_normal \
    --rna \
    --processes 16 \
    --fishers-method-context 2 \
    --minimum-test-reads 20 \
    --single-read-threshold -1.5 1.5 \
    --coverage-dampen-counts 2 0 \
    --per-read-statistics-basename tombo_normal_per_read \
    --multiprocess-region-size 200000
tombo detect_modifications de_novo --fast5-basedirs /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single \
--statistics-file-basename /root/sunxh/WaveCrossMamba/contrast/m6A \
--per-read-statistics-basename /root/sunxh/WaveCrossMamba/contrast/m6A  \
--processes 36
 """

' tombo preprocess annotate_raw_with_fastq     --fast5-basedir /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy_single     --fastq-filenames /root/sunxh/WaveCrossMamba/contrast/IVT_normalA.fastq     --overwrite\ntombo detect_modifications de_novo     --fast5-basedir /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single  --statistics-file-basename tombo_m6A --rna --processes 16\ntombo stats display_stats --per-read-statistics /root/tombo_m6A.tombo.stats.per_read_stats\ntombo detect_modifications de_novo     --fast5-basedir /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single     --statistics-file-basename tombo_normal     --rna     --processes 16     --fishers-method-context 2     --minimum-test-reads 20     --single-read-threshold -1.5 1.5     --coverage-dampen-counts 2 0     --per-read-statistics-basename tombo_normal_per_read     --multiprocess-region-size 200000\ntombo detect_modifications de_novo --fast5-basedirs /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single 

In [ ]:
#calculate ROC and PR
fprs,tprs,precisions,recalls,normalization_roc,normalization_pr=[],[],[],[],[],[]
from sklearn.metrics import roc_curve,auc,roc_auc_score,precision_recall_curve


from tombo import tombo_helper, tombo_stats, resquiggle
sample_per_read_stats_normal = tombo_stats.PerReadStats('/root/sunxh/WaveCrossMamba/contrast/normal.tombo.per_read_stats')
sample_per_read_stats_m6A = tombo_stats.PerReadStats('/root/sunxh/WaveCrossMamba/contrast/m6A.tombo.per_read_stats')



mod_dict_normal={"A1":{},"A2":{}}

reg_data = tombo_helper.intervalData(chrm="A1", start=1, end=2000, strand='+')
reg_per_read_stats_normal = sample_per_read_stats_normal.get_region_per_read_stats(reg_data)
print(f"染色体 A1 数据加载成功，共 {len(reg_per_read_stats_normal)} 条记录")


for item in reg_per_read_stats_normal:
    pos=item[0]
    probability=item[1]
    if pos not in mod_dict_normal["A1"]:
        mod_dict_normal["A1"][pos]=[]
    mod_dict_normal["A1"][pos].append(probability)
    

mod_dict_m6A={"A1":{},"A2":{}}

reg_data = tombo_helper.intervalData(chrm="A1", start=1, end=2000, strand='+')
reg_per_read_stats_m6A = sample_per_read_stats_m6A.get_region_per_read_stats(reg_data)
print(f"染色体 A1 数据加载成功，共 {len(reg_per_read_stats_m6A)} 条记录")


for item in reg_per_read_stats_m6A:
    pos=item[0]
    probability=item[1]
    if pos not in mod_dict_m6A["A1"]:
        mod_dict_m6A["A1"][pos]=[]
    mod_dict_m6A["A1"][pos].append(probability)

    



染色体 A1 数据加载成功，共 71583076 条记录
染色体 A1 数据加载成功，共 71583076 条记录


In [ ]:
probability=[]
label=[]

ref_dict={}
ref_dict = {}
first_contig = None 
first_sequence = "" 

with open("/root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"): 
            if first_contig: 
                break
            first_contig = line[1:]  
        else:
            first_sequence += line  


if first_contig:
    ref_dict[first_contig] = first_sequence

print("Extracted first chromosome:", first_contig)
print("Sequence length:", len(first_sequence))


DRACH_motifs=["AAACA","AGACA","GAACA","GGACA","TAACA","TGACA",
       "AAACC","AGACC","GAACC","GGACC","TAACC","TGACC",
       "AAACT","AGACT","GAACT","GGACT","TAACT","TGACT"]

count=0

for contig in ref_dict:
    sequence=ref_dict[contig]

    for i in range(2,len(sequence)-3):
        motif=sequence[i-2:i+3]
        if motif in DRACH_motifs:
            probability_list=mod_dict_normal[contig][i+2]
            for j in range(int(len(probability_list)/100)):
                if count>2000:
                    break
                probability.append(1-probability_list[int(j*100)])
                label.append(0)
                count+=1
                
count=0
for contig in ref_dict:
    sequence=ref_dict[contig]

    for i in range(2,len(sequence)-3):
        motif=sequence[i-2:i+3]
        if motif in DRACH_motifs:
            probability_list=mod_dict_m6A[contig][i+2]
            for j in range(int(len(probability_list)/100)):
                if count>2000:
                    break
                probability.append(1-probability_list[int(j*100)])
                label.append(1)
                count+=1
                
fpr, tpr, thersholds = roc_curve(label, probability)
precision, recall, thersholds = precision_recall_curve(label, probability)

fprs.extend(fpr)
tprs.extend(tpr)
precisions.extend(precision)
recalls.extend(recall)



roc_auc = round(auc(fpr, tpr), 3)
pr_auc = round(auc(recall, precision), 3)


roc_auc_str = "{:.3f}".format(roc_auc)
pr_auc_str = "{:.3f}".format(pr_auc)

print("Final roc_auc:", roc_auc_str)  
print("Final pr_auc:", pr_auc_str)


normalization_roc.extend([f"tombo AUC {roc_auc_str}"] * len(fpr))
normalization_pr.extend([f"tombo AUC {pr_auc_str}"] * len(precision))

Extracted first chromosome: A1
Sequence length: 1358
Final roc_auc: 0.779
Final pr_auc: 0.588
['tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78', 'tombo AUC 0.78']
['tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59', 'tombo AUC 0.59']


In [25]:
import pandas as pd
df_roc = pd.DataFrame({"FPR": fprs, "TPR": tprs, "AUC": normalization_roc})
df_pr = pd.DataFrame({"Precision": precisions, "Recall": recalls, "AUC": normalization_pr})
df_roc.to_csv("tombo_roc_drach.csv", sep=",", index=False)  
df_pr.to_csv("tombo_pr_drach.csv", sep=",", index=False)    

In [47]:
fprs,tprs,precisions,recalls,normalization_roc,normalization_pr=[],[],[],[],[],[]

probability=[]
label=[]

ref_dict={}
ref_dict = {}
first_contig = None 
first_sequence = "" 

with open("/root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"): 
            if first_contig: 
                break
            first_contig = line[1:] 
        else:
            first_sequence += line 


if first_contig:
    ref_dict[first_contig] = first_sequence

print("Extracted first chromosome:", first_contig)
print("Sequence length:", len(first_sequence))

#RRACH
RRACH_motifs=["AAACA","AGACA","GAACA","GGACA",
       "AAACC","AGACC","GAACC","GGACC",
       "AAACT","AGACT","GAACT","GGACT",]

count=0
for contig in ref_dict:
    sequence=ref_dict[contig]

    for i in range(2,len(sequence)-3):
        motif=sequence[i-2:i+3]
        if motif in RRACH_motifs:
            probability_list=mod_dict_normal[contig][i+2]
            for j in range(int(len(probability_list)/100)):
                if count>2000:
                    break
                probability.append(1-probability_list[int(j*100)])
                label.append(0)
                count+=1
                
count=0
for contig in ref_dict:
    sequence=ref_dict[contig]

    for i in range(2,len(sequence)-3):
        motif=sequence[i-2:i+3]
        if motif in RRACH_motifs:
            probability_list=mod_dict_m6A[contig][i+2]
            for j in range(int(len(probability_list)/100)):
                if count>2000:
                    break
                probability.append(1-probability_list[int(j*100)])
                label.append(1)
                count+=1
                
fpr,tpr,thersholds=roc_curve(label,probability)
precision,recall,thersholds=precision_recall_curve(label,probability)
roc_auc=auc(fpr,tpr)
pr_auc=auc(recall,precision)



fprs.extend(fpr)
tprs.extend(tpr)
precisions.extend(precision)
recalls.extend(recall)
normalization_roc.extend(["tombo AUC %.2f" %roc_auc]*len(fpr))
normalization_pr.extend(["tombo AUC %.2f" %pr_auc]*len(precision))
print(roc_auc)

Extracted first chromosome: A1
Sequence length: 1358
0.6403132588057368


In [40]:
possible_chromosomes = ["A1", "A2", "C1", "C2", "G1", "G2", "U1", "U2", "NE"]

existing_chromosomes = []

for chrm in possible_chromosomes:
    reg_data = tombo_helper.intervalData(chrm=chrm, start=1, end=2000, strand='+')
    reg_per_read_stats = sample_per_read_stats_normal.get_region_per_read_stats(reg_data)
    
    if reg_per_read_stats is not None:
        existing_chromosomes.append(chrm)

print("Detected chromosomes in m6A data:", existing_chromosomes)


/root/miniconda3/envs/nanopore/lib/python3.6/site-packages/tombo/tombo_stats.py:3400: H5pyDeprecationWarning: dataset.value has been deprecated. Use dataset[()] instead.
  zip(self.per_read_blocks[block_name]['read_ids'].value,
/root/miniconda3/envs/nanopore/lib/python3.6/site-packages/tombo/tombo_stats.py:3401: H5pyDeprecationWarning: dataset.value has been deprecated. Use dataset[()] instead.
  self.per_read_blocks[block_name]['read_id_vals'].value)])


Detected chromosomes in m6A data: ['A1', 'U1', 'U2']


In [45]:
import pandas as pd
df_roc = pd.DataFrame({"FPR": fprs, "TPR": tprs, "AUC": normalization_roc})
df_pr = pd.DataFrame({"Precision": precisions, "Recall": recalls, "AUC": normalization_pr})
df_roc.to_csv("tombo_roc_RRACH.csv", sep=",", index=False)  
df_pr.to_csv("tombo_pr_RRACH.csv", sep=",", index=False)    
